# Mini Projeto — Análise de Vendas de Halloween

Este projeto tem como objetivo analisar uma base fictícia de vendas de produtos relacionados ao Halloween.

Ao longo do projeto serão realizadas etapas de leitura dos dados, limpeza, validação, tratamento de datas e análise de vendas, buscando identificar padrões que possam apoiar decisões de negócio.

## Sprint 1 — Configuração e leitura dos dados

In [2]:
# Importar a biblioteca Pandas para trabalhar com os dados.

import pandas as pd

In [3]:
# Carregar a base de vendas preservando os dados inicialmente como texto.

df = pd.read_csv(
    "halloween_vendas.csv",
    sep=",",
    encoding="utf-8",
    dtype=str
)

In [4]:
# Conferir a quantidade de registros e visualizar as primeiras linhas.

print("Total de registros:", len(df))

display(df.head(3))

Total de registros: 1500


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [5]:
# Verificar os nomes das colunas disponíveis na base.

print(df.columns.tolist())

['pedido_id', 'data_venda', 'produto', 'categoria', 'canal_venda', 'cidade', 'quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'status_pedido', 'publico', 'tamanho', 'data_entrega']


In [5]:
# Conferir a quantidade de valores ausentes em cada coluna.

print(df.isna().sum())

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64


In [6]:
# Verificar como os valores da coluna categoria aparecem na base.

print(df["categoria"].value_counts(dropna=False))

categoria
Fantasia            892
Acessorio           459
 Fantasia            48
  FANTASIA!!!        39
 Acessorio           28
NaN                  18
  ACESSORIO!!!       16
Name: count, dtype: int64


In [7]:
# Verificar como os valores da coluna canal_venda aparecem na base.

print(df["canal_venda"].value_counts(dropna=False))

canal_venda
Web              646
Loja física      535
Mercado          274
 web              20
 loja física      16
 mercado           9
Name: count, dtype: int64


## Sprint 2 — Limpeza e padronização dos textos

In [10]:
# Criar uma função para padronizar os valores da coluna categoria.

import re

def limpar_categoria(valor):
    if pd.isna(valor):
        return "sem categoria"

    valor = valor.strip()
    valor = valor.lower()
    valor = re.sub(r"[^a-zà-ÿ\s]", "", valor)

    return valor

In [11]:
# Aplicar a limpeza da categoria em uma nova coluna.

df["categoria_limpa"] = df["categoria"].apply(limpar_categoria)

In [12]:
# Conferir o resultado da padronização da categoria.

print(df["categoria_limpa"].value_counts(dropna=False))

categoria_limpa
fantasia         979
acessorio        503
sem categoria     18
Name: count, dtype: int64


In [13]:
# Criar uma função para padronizar os valores da coluna canal_venda.

def limpar_canal_venda(valor):
    valor = valor.strip()
    valor = valor.lower()

    if valor == "web":
        return "Web"
    elif valor == "loja física":
        return "Loja física"
    elif valor == "mercado":
        return "Mercado"

    return valor

In [14]:
# Aplicar a limpeza do canal de venda em uma nova coluna e conferir o resultado da padronização.

df["canal_venda_limpo"] = df["canal_venda"].apply(limpar_canal_venda)
print(df["canal_venda_limpo"].value_counts(dropna=False))

canal_venda_limpo
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64


In [15]:
# Contar quantos registros da categoria estavam diferentes do padrão esperado e 
# também quantos registros do canal de venda foram ajustados.

categorias_validas = ["Fantasia", "Acessorio"]

problema_categoria = ~df["categoria"].isin(categorias_validas)

quantidade_inconsistencias_categoria = problema_categoria.sum()

print(
    "Registros com inconsistências na categoria:",
    quantidade_inconsistencias_categoria
)


quantidade_corrigida_canal = (
    df["canal_venda"] != df["canal_venda_limpo"]
).sum()

print(
    "Registros ajustados em canal_venda:",
    quantidade_corrigida_canal
)

Registros com inconsistências na categoria: 149
Registros ajustados em canal_venda: 45


## Sprint 3 — Validação dos dados numéricos

In [16]:
# Converter quantidade e preço para formato numérico em novas colunas.

df["quantidade_num"] = pd.to_numeric(
    df["quantidade"],
    errors="coerce"
)

df["preco_unitario_num"] = pd.to_numeric(
    df["preco_unitario"],
    errors="coerce"
)

In [17]:
# Identificar registros com quantidade ou preço inválidos.

df["problema_quantidade"] = (
    df["quantidade_num"].isna()
    | (df["quantidade_num"] <= 0)
    | (df["quantidade_num"] % 1 != 0)
)

df["problema_preco"] = (
    df["preco_unitario_num"].isna()
    | (df["preco_unitario_num"] < 0)
)

In [18]:
# Contar quantos registros apresentam problemas numéricos.

print(
    "Problemas em quantidade:",
    df["problema_quantidade"].sum()
)

print(
    "Problemas em preço:",
    df["problema_preco"].sum()
)

Problemas em quantidade: 7
Problemas em preço: 12


In [19]:
# Identificar os pedidos que possuem algum problema de quantidade ou preço.

pedidos_com_problema = df[
    df["problema_quantidade"] | df["problema_preco"]
]

quantidade_pedidos_afetados = pedidos_com_problema["pedido_id"].nunique()

print(
    "Pedidos únicos afetados por problemas numéricos:",
    quantidade_pedidos_afetados
)

Pedidos únicos afetados por problemas numéricos: 19


In [20]:
# Visualizar os registros com problemas numéricos.

display(
    pedidos_com_problema[
        [
            "pedido_id",
            "quantidade",
            "preco_unitario",
            "problema_quantidade",
            "problema_preco"
        ]
    ]
)

,pedido_id,quantidade,preco_unitario,problema_quantidade,problema_preco
23,1500,3,NaN,False,True
164,630,-1,22,True,False
199,625,2,NaN,False,True
216,1000,1,NaN,False,True
239,1250,1,NaN,False,True
293,500,1,NaN,False,True
503,375,3,NaN,False,True
571,250,4,NaN,False,True
583,125,1,NaN,False,True
672,840,-1,23.32,True,False


## Sprint 4 — Regras de negócio e tratamento de datas

In [21]:
# Separar os pedidos sem data de entrega para testar a hipótese.

sem_entrega = df[df["data_entrega"].isna()]

sem_entrega_cancelados = sem_entrega[
    sem_entrega["status_pedido"] == "Cancelado"
]

sem_entrega_outros_status = sem_entrega[
    sem_entrega["status_pedido"] != "Cancelado"
]

In [22]:
# Comparar quantos pedidos sem entrega estão cancelados e quantos possuem outro status.

print("Sem entrega e cancelados:", len(sem_entrega_cancelados))
print("Sem entrega com outro status:", len(sem_entrega_outros_status))

Sem entrega e cancelados: 109
Sem entrega com outro status: 34


In [23]:
# Verificar se a hipótese é verdadeira ou falsa.

if len(sem_entrega_outros_status) == 0:
    print("Hipótese verdadeira.")
else:
    print("Hipótese falsa.")

#Falsa porque existem pedidos sem data de entrega que não estão cancelados.

Hipótese falsa.


In [24]:
# Mostrar alguns pedidos sem data de entrega que não estão cancelados.

display(
    sem_entrega_outros_status[
        [
            "pedido_id",
            "status_pedido",
            "data_entrega"
        ]
    ].head(3)
)

,pedido_id,status_pedido,data_entrega
49,1170,Concluído,NaN
117,990,Concluído,NaN
152,740,Concluído,NaN


In [25]:
# Importar datetime para converter os textos de data em datas reais.

from datetime import datetime

In [26]:
# Criar uma função para converter data_venda para o formato de data.

def converter_data_venda(valor):
    try:
        return datetime.strptime(
            valor,
            "%Y-%m-%d %H:%M:%S"
        )
    except (ValueError, TypeError):
        return None

In [27]:
# Aplicar a conversão em uma nova coluna e contar quantas datas de venda são inválidas.

df["data_venda_convertida"] = df["data_venda"].apply(
    converter_data_venda
)
datas_invalidas = df["data_venda_convertida"].isna().sum()

print("Datas de venda inválidas:", datas_invalidas)

Datas de venda inválidas: 4


In [28]:
# Criar colunas de ano, mês, dia e hora a partir da data convertida.

df["ano"] = df["data_venda_convertida"].apply(
    lambda x: x.year if pd.notna(x) else None
)

df["mes"] = df["data_venda_convertida"].apply(
    lambda x: x.month if pd.notna(x) else None
)

df["dia"] = df["data_venda_convertida"].apply(
    lambda x: x.day if pd.notna(x) else None
)

df["hora"] = df["data_venda_convertida"].apply(
    lambda x: x.hour if pd.notna(x) else None
)

In [29]:
# Criar uma coluna no formato ano-mês para agrupar os pedidos por período.

df["ano_mes"] = df["data_venda_convertida"].apply(
    lambda x: x.strftime("%Y-%m") if pd.notna(x) else None
)

# Conferir as novas colunas criadas a partir da data.

display(
    df[
        [
            "data_venda",
            "data_venda_convertida",
            "ano",
            "mes",
            "dia",
            "hora",
            "ano_mes"
        ]
    ].head()
)

,data_venda,data_venda_convertida,ano,mes,dia,hora,ano_mes
0,2025-11-21 18:02:35,2025-11-21 18:02:35,2025.0,11.0,21.0,18.0,2025-11
1,2024-10-29 18:41:26,2024-10-29 18:41:26,2024.0,10.0,29.0,18.0,2024-10
2,2025-09-05 21:16:09,2025-09-05 21:16:09,2025.0,9.0,5.0,21.0,2025-09
3,2024-11-08 11:29:44,2024-11-08 11:29:44,2024.0,11.0,8.0,11.0,2024-11
4,2025-08-23 11:22:05,2025-08-23 11:22:05,2025.0,8.0,23.0,11.0,2025-08


In [30]:
# Mostrar os registros cuja data de venda não pôde ser convertida.

registros_datas_invalidas = df[
    df["data_venda_convertida"].isna()
]

display(
    registros_datas_invalidas[
        [
            "pedido_id",
            "data_venda"
        ]
    ]
)

,pedido_id,data_venda
152,740,data_invalida
255,370,data_invalida
714,1480,data_invalida
884,1110,data_invalida


In [31]:
# Conferir quantos registros possuem data válida e inválida.

total_registros = len(df)

registros_datas_validas = df[
    "data_venda_convertida"
].notna().sum()

print("Total de registros:", total_registros)
print("Registros com data válida:", registros_datas_validas)
print("Registros com data inválida:", datas_invalidas)

Total de registros: 1500
Registros com data válida: 1496
Registros com data inválida: 4


## Sprint 5 — Análises de negócio

In [32]:
# Contar quantos pedidos existem em cada período de ano e mês.

pedidos_por_periodo = (
    df[df["data_venda_convertida"].notna()]
    ["ano_mes"]
    .value_counts()
    .sort_index()
)

print("Pedidos por período:")
print(pedidos_por_periodo)

Pedidos por período:
ano_mes
2024-08     77
2024-09    104
2024-10    393
2024-11    174
2025-08     62
2025-09     94
2025-10    410
2025-11    182
Name: count, dtype: int64


In [33]:
# Identificar o período com maior quantidade de pedidos.

periodo_maior_pedidos = pedidos_por_periodo.idxmax()
maior_quantidade_pedidos = pedidos_por_periodo.max()

print("Período com mais pedidos:", periodo_maior_pedidos)
print("Quantidade de pedidos:", maior_quantidade_pedidos)

Período com mais pedidos: 2025-10
Quantidade de pedidos: 410


In [34]:
# Comparar a quantidade de pedidos entre outubro de 2024 e outubro de 2025.

outubro_2024 = pedidos_por_periodo["2024-10"]
outubro_2025 = pedidos_por_periodo["2025-10"]

diferenca_outubro = outubro_2025 - outubro_2024

print("Pedidos em outubro de 2024:", outubro_2024)
print("Pedidos em outubro de 2025:", outubro_2025)
print("Diferença entre os períodos:", diferenca_outubro)

Pedidos em outubro de 2024: 393
Pedidos em outubro de 2025: 410
Diferença entre os períodos: 17


In [35]:
# Calcular o crescimento percentual de outubro de 2024 para outubro de 2025.

crescimento_percentual_outubro = (
    (outubro_2025 - outubro_2024)
    / outubro_2024
    * 100
)

print(
    "Crescimento percentual:",
    round(crescimento_percentual_outubro, 2),
    "%"
)

Crescimento percentual: 4.33 %


In [36]:
# Contar quantos pedidos existem em cada canal de venda.

pedidos_por_canal = (
    df["canal_venda_limpo"]
    .value_counts()
)

print("Pedidos por canal de venda:")
print(pedidos_por_canal)

Pedidos por canal de venda:
canal_venda_limpo
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64


In [37]:
# Calcular a participação percentual de cada canal no total de pedidos.

percentual_por_canal = (
    pedidos_por_canal
    / pedidos_por_canal.sum()
    * 100
).round(2)

print("Participação percentual por canal:")
print(percentual_por_canal)

Participação percentual por canal:
canal_venda_limpo
Web            44.40
Loja física    36.73
Mercado        18.87
Name: count, dtype: float64


In [38]:
# Cruzar período e canal para contar quantos pedidos cada canal teve em cada mês.

pedidos_periodo_canal = (
    df[df["data_venda_convertida"].notna()]
    .groupby(["ano_mes", "canal_venda_limpo"])
    .size()
    .unstack(fill_value=0)
)

display(pedidos_periodo_canal)

canal_venda_limpo,Loja física,Mercado,Web
ano_mes,,,
2024-08,27,17,33
2024-09,26,22,56
2024-10,136,74,183
2024-11,89,38,47
2025-08,18,16,28
2025-09,24,13,57
2025-10,130,69,211
2025-11,101,34,47


In [39]:
# Identificar qual canal teve mais pedidos em cada período.

canal_lider_por_periodo = pedidos_periodo_canal.idxmax(axis=1)

print("Canal líder por período:")
print(canal_lider_por_periodo)

Canal líder por período:
ano_mes
2024-08            Web
2024-09            Web
2024-10            Web
2024-11    Loja física
2025-08            Web
2025-09            Web
2025-10            Web
2025-11    Loja física
dtype: str


In [40]:
# Converter o desconto para número para calcular o valor final de cada venda.

df["desconto_pct_num"] = pd.to_numeric(
    df["desconto_pct"],
    errors="coerce"
)

In [41]:
# Verificar se existem descontos que não puderam ser convertidos.

descontos_invalidos = df["desconto_pct_num"].isna().sum()

print(
    "Descontos inválidos:",
    descontos_invalidos
)

Descontos inválidos: 0


In [45]:
# Calcular o valor de cada venda considerando quantidade, preço e desconto.

df["valor_venda"] = (
    df["quantidade_num"]
    * df["preco_unitario_num"]
    * (1 - df["desconto_pct_num"].fillna(0) / 100)
)

In [47]:
# Somar o faturamento de cada canal em cada período.

faturamento_periodo_canal = (
    df[
        (df["data_venda_convertida"].notna())
        & (~df["problema_quantidade"])
        & (~df["problema_preco"])
        & (df["status_pedido"] == "Concluído")
    ]
    .groupby(["ano_mes", "canal_venda_limpo"])["valor_venda"]
    .sum()
    .unstack(fill_value=0)
    .round(2)
)

display(faturamento_periodo_canal)

canal_venda_limpo,Loja física,Mercado,Web
ano_mes,,,
2024-08,3615.45,3522.75,3437.30
2024-09,3037.15,4134.25,8006.65
2024-10,17754.10,11878.10,27579.25
2024-11,10472.70,5996.00,5912.00
2025-08,2705.17,3056.03,4100.03
2025-09,2735.81,4289.71,8031.09
2025-10,19021.22,12379.48,32063.68
2025-11,12513.56,6166.60,5966.42


In [48]:
# Identificar o canal com maior faturamento em cada período.

canal_maior_faturamento = faturamento_periodo_canal.idxmax(axis=1)

maior_faturamento_periodo = faturamento_periodo_canal.max(axis=1)

resultado_faturamento = pd.DataFrame({
    "canal_maior_faturamento": canal_maior_faturamento,
    "valor_faturado": maior_faturamento_periodo
})

display(resultado_faturamento)

,canal_maior_faturamento,valor_faturado
ano_mes,,
2024-08,Loja física,3615.45
2024-09,Web,8006.65
2024-10,Web,27579.25
2024-11,Loja física,10472.70
2025-08,Web,4100.03
2025-09,Web,8031.09
2025-10,Web,32063.68
2025-11,Loja física,12513.56


In [49]:
# Adicionar o faturamento total de cada período.

faturamento_periodo_canal["Total"] = (
    faturamento_periodo_canal.sum(axis=1)
)

display(faturamento_periodo_canal)

canal_venda_limpo,Loja física,Mercado,Web,Total
ano_mes,,,,
2024-08,3615.45,3522.75,3437.30,10575.50
2024-09,3037.15,4134.25,8006.65,15178.05
2024-10,17754.10,11878.10,27579.25,57211.45
2024-11,10472.70,5996.00,5912.00,22380.70
2025-08,2705.17,3056.03,4100.03,9861.23
2025-09,2735.81,4289.71,8031.09,15056.61
2025-10,19021.22,12379.48,32063.68,63464.38
2025-11,12513.56,6166.60,5966.42,24646.58


In [50]:
# Comparar o canal com mais pedidos ao canal com maior faturamento em cada período.

comparacao_canais = pd.DataFrame({
    "mais_pedidos": canal_lider_por_periodo,
    "maior_faturamento": canal_maior_faturamento,
    "faturamento_total": faturamento_periodo_canal["Total"]
})

display(comparacao_canais)

,mais_pedidos,maior_faturamento,faturamento_total
ano_mes,,,
2024-08,Web,Loja física,10575.50
2024-09,Web,Web,15178.05
2024-10,Web,Web,57211.45
2024-11,Loja física,Loja física,22380.70
2025-08,Web,Web,9861.23
2025-09,Web,Web,15056.61
2025-10,Web,Web,63464.38
2025-11,Loja física,Loja física,24646.58


In [51]:
# Calcular o ticket médio por período e canal.

ticket_medio_periodo_canal = (
    df[
        (df["data_venda_convertida"].notna())
        & (~df["problema_quantidade"])
        & (~df["problema_preco"])
        & (df["status_pedido"] == "Concluído")
    ]
    .groupby(["ano_mes", "canal_venda_limpo"])["valor_venda"]
    .mean()
    .unstack()
    .round(2)
)

display(ticket_medio_periodo_canal)

canal_venda_limpo,Loja física,Mercado,Web
ano_mes,,,
2024-08,150.64,220.17,149.45
2024-09,132.05,196.87,166.81
2024-10,142.03,188.54,169.20
2024-11,126.18,162.05,151.59
2025-08,180.34,191.00,186.36
2025-09,136.79,329.98,174.59
2025-10,158.51,182.05,178.13
2025-11,139.04,205.55,157.01


In [52]:
# Identificar o canal com maior ticket médio em cada período.

canal_maior_ticket = ticket_medio_periodo_canal.idxmax(axis=1)

maior_ticket_periodo = ticket_medio_periodo_canal.max(axis=1)

resultado_ticket = pd.DataFrame({
    "canal_maior_ticket": canal_maior_ticket,
    "maior_ticket_medio": maior_ticket_periodo
})

display(resultado_ticket)

,canal_maior_ticket,maior_ticket_medio
ano_mes,,
2024-08,Mercado,220.17
2024-09,Mercado,196.87
2024-10,Mercado,188.54
2024-11,Mercado,162.05
2025-08,Mercado,191.00
2025-09,Mercado,329.98
2025-10,Mercado,182.05
2025-11,Mercado,205.55


In [53]:
# Converter o custo unitário para formato numérico.

df["custo_unitario_num"] = pd.to_numeric(
    df["custo_unitario"],
    errors="coerce"
)

print(
    "Custos inválidos:",
    df["custo_unitario_num"].isna().sum()
)

Custos inválidos: 7


In [54]:
# Calcular a margem bruta das vendas com dados válidos.

df["margem_bruta"] = (
    df["valor_venda"]
    - (df["quantidade_num"] * df["custo_unitario_num"])
)

In [55]:
# Selecionar apenas vendas válidas para analisar a margem.

vendas_margem_validas = df[
    (df["data_venda_convertida"].notna())
    & (~df["problema_quantidade"])
    & (~df["problema_preco"])
    & (df["custo_unitario_num"].notna())
    & (df["status_pedido"] == "Concluído")
]

In [56]:
# Somar a margem bruta por período e canal de venda.

margem_periodo_canal = (
    vendas_margem_validas
    .groupby(["ano_mes", "canal_venda_limpo"])["margem_bruta"]
    .sum()
    .unstack(fill_value=0)
    .round(2)
)

display(margem_periodo_canal)

canal_venda_limpo,Loja física,Mercado,Web
ano_mes,,,
2024-08,1694.45,1904.75,1601.30
2024-09,1477.15,2298.25,3769.65
2024-10,8566.10,6614.10,12615.00
2024-11,4412.70,3110.00,2320.00
2025-08,1228.59,1763.89,1920.67
2025-09,1320.71,2153.81,3748.69
2025-10,9058.28,6729.68,15024.17
2025-11,5058.58,3002.50,2259.07


In [57]:
# Calcular a margem percentual de cada canal em cada período.

margem_percentual = (
    margem_periodo_canal
    / faturamento_periodo_canal.drop(columns="Total")
    * 100
).round(2)

display(margem_percentual)

canal_venda_limpo,Loja física,Mercado,Web
ano_mes,,,
2024-08,46.87,54.07,46.59
2024-09,48.64,55.59,47.08
2024-10,48.25,55.68,45.74
2024-11,42.14,51.87,39.24
2025-08,45.42,57.72,46.85
2025-09,48.27,50.21,46.68
2025-10,47.62,54.36,46.86
2025-11,40.42,48.69,37.86


In [58]:
# Identificar o canal com maior margem em cada período.

canal_maior_margem = margem_periodo_canal.idxmax(axis=1)

maior_margem_periodo = margem_periodo_canal.max(axis=1)

resultado_margem = pd.DataFrame({
    "canal_maior_margem": canal_maior_margem,
    "maior_margem": maior_margem_periodo
})

display(resultado_margem)

,canal_maior_margem,maior_margem
ano_mes,,
2024-08,Mercado,1904.75
2024-09,Web,3769.65
2024-10,Web,12615.00
2024-11,Loja física,4412.70
2025-08,Web,1920.67
2025-09,Web,3748.69
2025-10,Web,15024.17
2025-11,Loja física,5058.58


In [59]:
# Reunir os principais indicadores de negócio por período.

resumo_negocio = pd.DataFrame({
    "mais_pedidos": canal_lider_por_periodo,
    "maior_faturamento": canal_maior_faturamento,
    "maior_ticket": canal_maior_ticket,
    "maior_margem": canal_maior_margem,
    "maior_margem_percentual": margem_percentual.idxmax(axis=1),
    "faturamento_total": faturamento_periodo_canal["Total"]
})

display(resumo_negocio)

,mais_pedidos,maior_faturamento,maior_ticket,maior_margem,maior_margem_percentual,faturamento_total
ano_mes,,,,,,
2024-08,Web,Loja física,Mercado,Mercado,Mercado,10575.50
2024-09,Web,Web,Mercado,Web,Mercado,15178.05
2024-10,Web,Web,Mercado,Web,Mercado,57211.45
2024-11,Loja física,Loja física,Mercado,Loja física,Mercado,22380.70
2025-08,Web,Web,Mercado,Web,Mercado,9861.23
2025-09,Web,Web,Mercado,Web,Mercado,15056.61
2025-10,Web,Web,Mercado,Web,Mercado,63464.38
2025-11,Loja física,Loja física,Mercado,Loja física,Mercado,24646.58


In [60]:
# Comparar o faturamento de cada mês entre 2024 e 2025.

faturamento_2024 = faturamento_periodo_canal["Total"].loc[
    ["2024-08", "2024-09", "2024-10", "2024-11"]
]

faturamento_2025 = faturamento_periodo_canal["Total"].loc[
    ["2025-08", "2025-09", "2025-10", "2025-11"]
]

crescimento_faturamento = (
    (faturamento_2025.values - faturamento_2024.values)
    / faturamento_2024.values
    * 100
)

comparacao_faturamento = pd.DataFrame({
    "mes": ["Agosto", "Setembro", "Outubro", "Novembro"],
    "faturamento_2024": faturamento_2024.values,
    "faturamento_2025": faturamento_2025.values,
    "crescimento_percentual": crescimento_faturamento.round(2)
})

display(comparacao_faturamento)

,mes,faturamento_2024,faturamento_2025,crescimento_percentual
0,Agosto,10575.50,9861.23,-6.75
1,Setembro,15178.05,15056.61,-0.80
2,Outubro,57211.45,63464.38,10.93
3,Novembro,22380.70,24646.58,10.12


In [61]:
# Identificar os principais destaques gerais da análise.

periodo_maior_faturamento = faturamento_periodo_canal["Total"].idxmax()

valor_maior_faturamento = faturamento_periodo_canal["Total"].max()

canal_melhor_margem_percentual = (
    margem_percentual.mean()
    .idxmax()
)

media_margem_percentual = (
    margem_percentual.mean()
    .max()
)

print("Período com maior faturamento:", periodo_maior_faturamento)

print(
    "Maior faturamento:",
    round(valor_maior_faturamento, 2)
)

print(
    "Canal com maior margem percentual média:",
    canal_melhor_margem_percentual
)

print(
    "Margem percentual média:",
    round(media_margem_percentual, 2),
    "%"
)

Período com maior faturamento: 2025-10
Maior faturamento: 63464.38
Canal com maior margem percentual média: Mercado
Margem percentual média: 53.52 %
